# DSCI 552 Homework 3

**Name:** Shirley(Xue) Feng  
**GitHub Username:** snowxxxue 
**USC ID:** 7145149057

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### 1(a) Download and Load the AReM Dataset

In [1]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

import numpy as np
import pandas as pd

In [2]:
zip_path = Path("AReM.zip")
data_dir = Path("AReM")

data_dir.mkdir(exist_ok=True)

with ZipFile(zip_path, "r") as zip_file:
    for member in zip_file.infolist():
        if member.filename != "/" and not Path(member.filename).is_absolute():
            zip_file.extract(member, data_dir)

print("Dataset extracted to:", data_dir)

Dataset extracted to: AReM


In [3]:
csv_files = sorted(data_dir.glob("*/*.csv"))

print("Number of CSV files:", len(csv_files))

Number of CSV files: 88


### 1(b) Train-Test Split

In [4]:
train_files = []
test_files = []

for file in csv_files:
    activity = file.parent.name
    dataset_number = int(file.stem.replace("dataset", ""))

    if activity in ["bending1", "bending2"]:
        if dataset_number in [1, 2]:
            test_files.append(file)
        else:
            train_files.append(file)
    else:
        if dataset_number in [1, 2, 3]:
            test_files.append(file)
        else:
            train_files.append(file)

print("Number of training instances:", len(train_files))
print("Number of test instances:", len(test_files))

Number of training instances: 69
Number of test instances: 19


In [5]:
split_records = []

for file in train_files:
    split_records.append({
        "activity": file.parent.name,
        "set": "train"
    })

for file in test_files:
    split_records.append({
        "activity": file.parent.name,
        "set": "test"
    })

split_summary = pd.DataFrame(split_records)

pd.crosstab(
    split_summary["activity"],
    split_summary["set"]
)

set,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Datasets 1 and 2 in the bending1 and bending2 folders were assigned
to the test set. Datasets 1, 2, and 3 in the other activity folders
were also assigned to the test set. All remaining datasets were assigned
to the training set. The training set contains 69 instances, and the
test set contains 19 instances.

### 1(c) Feature Extraction

#### 1(c)(i) Common Time-Domain Features

Common time-domain features used in time-series classification include:

- Minimum and maximum
- Range and peak-to-peak amplitude
- Mean and median
- Variance and standard deviation
- First quartile and third quartile
- Interquartile range
- Mean absolute deviation
- Root mean square
- Signal energy
- Skewness
- Kurtosis
- Zero-crossing rate
- Autocorrelation
- Trend or slope

These features summarize different properties of a time series, such as its
central tendency, variability, distribution shape, magnitude, and temporal
behavior.

#### Reference

Lubba, C. H., Sethi, S. S., Knaute, P., Schultz, S. R.,
Fulcher, B. D., and Jones, N. S. (2019).
*catch22: CAnonical Time-series CHaracteristics*.
Data Mining and Knowledge Discovery, 33, 1821–1852.
https://doi.org/10.1007/s10618-019-00647-x

#### 1(c)(ii) Extraction of Time-Domain Features

In [6]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

time_series_names = column_names[1:]


def read_instance(file):
    data = pd.read_csv(
        file,
        skiprows=5,
        header=None,
        names=column_names,
        sep=r"[,\s]+",
        engine="python",
        usecols=range(7)
    )

    return data

In [7]:
activity_order = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

all_files = sorted(
    csv_files,
    key=lambda file: (
        activity_order.index(file.parent.name),
        int(file.stem.replace("dataset", ""))
    )
)

feature_rows = []

for instance_number, file in enumerate(all_files, start=1):
    data = read_instance(file)

    row = {
        "Instance": instance_number,
        "activity": file.parent.name,
        "dataset": file.stem,
        "set": "test" if file in test_files else "train"
    }

    for i, column in enumerate(time_series_names, start=1):
        values = pd.to_numeric(
            data[column],
            errors="coerce"
        ).dropna()

        row[f"min{i}"] = values.min()
        row[f"max{i}"] = values.max()
        row[f"mean{i}"] = values.mean()
        row[f"median{i}"] = values.median()
        row[f"std{i}"] = values.std(ddof=1)
        row[f"1st_quart{i}"] = values.quantile(0.25)
        row[f"3rd_quart{i}"] = values.quantile(0.75)

    feature_rows.append(row)

feature_dataset = pd.DataFrame(feature_rows)

In [8]:
feature_columns = [
    column
    for column in feature_dataset.columns
    if column not in ["Instance", "activity", "dataset", "set"]
]

assignment_feature_table = feature_dataset[
    ["Instance"] + feature_columns
]

print("Shape of feature table:", assignment_feature_table.shape)

assignment_feature_table

Shape of feature table: (88, 43)


,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


In [9]:
print("Number of instances:", feature_dataset.shape[0])
print("Number of extracted features:", len(feature_columns))
print(
    "Number of missing feature values:",
    feature_dataset[feature_columns].isna().sum().sum()
)

Number of instances: 88
Number of extracted features: 42
Number of missing feature values: 0


For each of the six time series in every instance, I extracted the
minimum, maximum, mean, median, sample standard deviation, first
quartile, and third quartile. This produced 42 time-domain features
for each instance. The resulting feature table contains 88 instances
and 42 extracted features.

#### 1(c)(iii) Standard Deviations and 90% Bootstrap Confidence Intervals

In [10]:
X = feature_dataset[feature_columns].to_numpy()

n_instances = X.shape[0]
n_features = X.shape[1]
n_bootstrap = 10000

rng = np.random.default_rng(42)

bootstrap_std = np.empty((n_bootstrap, n_features))

for i in range(n_bootstrap):
    sample_indices = rng.integers(
        low=0,
        high=n_instances,
        size=n_instances
    )

    bootstrap_sample = X[sample_indices]

    bootstrap_std[i] = bootstrap_sample.std(
        axis=0,
        ddof=1
    )

estimated_std = X.std(axis=0, ddof=1)

ci_lower = np.percentile(
    bootstrap_std,
    5,
    axis=0
)

ci_upper = np.percentile(
    bootstrap_std,
    95,
    axis=0
)

confidence_interval_table = pd.DataFrame({
    "Feature": feature_columns,
    "Standard Deviation": estimated_std,
    "90% CI Lower": ci_lower,
    "90% CI Upper": ci_upper
})

confidence_interval_table.round(3)

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.570,8.236,10.774
1,max1,4.394,3.340,5.293
2,mean1,5.336,4.704,5.876
3,median1,5.440,4.784,6.006
4,std1,1.772,1.569,1.950
5,1st_quart1,6.154,5.574,6.649
6,3rd_quart1,5.139,4.339,5.834
7,min2,0.000,0.000,0.000
8,max2,5.063,4.631,5.401
9,mean2,1.574,1.396,1.706


I estimated the sample standard deviation of each of the 42 extracted
time-domain features across the 88 instances. I then used 10,000
bootstrap samples, with sampling performed with replacement, to construct
90% confidence intervals. The lower and upper bounds were calculated
using the 5th and 95th percentiles of the bootstrap standard deviations.

#### 1(c)(iv) Selection of Important Time-Domain Features

I selected the minimum, mean, and maximum as the three most important
time-domain features. The mean represents the overall level of a time
series, while the minimum and maximum capture its lower and upper
extremes. Together, these three features provide a simple summary of
the central tendency and range of each time series.

## 2. ISLR 3.7.4

### (a)

I would expect the cubic model to have a lower or equal training RSS.
The cubic model is more flexible, and setting its quadratic and cubic
coefficients to zero gives the linear model. Therefore, it can fit the
training data at least as well as the linear model.

### (b)

I would expect the linear model to have a lower test RSS. Since the true
relationship is linear, the extra terms in the cubic model may fit noise
and lead to overfitting.

### (c)

I would expect the cubic model to have a lower or equal training RSS
because it is more flexible and includes the linear model as a special
case.

### (d)

There is not enough information to tell. If the true relationship is
close to linear, the linear model may perform better. If it is strongly
nonlinear and close to a cubic relationship, the cubic model may perform
better.